# Nettoyage & enrichissement des boxscores (minutes, DNP, totaux, ratios)

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from src.config import *
from src.utils import *
from src.feature_builder import *
from src.feature_aggregation import *

#display full columns
pd.set_option('display.max_column', None)
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_seq_items', None)
# pd.set_option('display.max_colwidth', 500)
# pd.set_option('expand_frame_repr', True)

In [2]:
#store start time of notebook
start_time = datetime.now()
print("Start time: ", start_time)

Start time:  2025-06-05 15:46:53.652941


# 🔁 Chargement des fichiers

In [3]:
boxscores_file = get_latest_file(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR)
games_file = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)
df_boxscores = pd.read_csv(boxscores_file, dtype={'gameId': str})
df_games = pd.read_csv(games_file, dtype={'GAME_ID': str})


In [4]:
df_boxscores

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage
0,0021100005,1610612746,Los Angeles,Clippers,LAC,clippers,2406,Caron,Butler,C. Butler,caron-butler,F,NaN,NaN,36:01,112.9,87.3,22.9,25.5,0.174,0.00,26.7,0.000,0.238,0.125,0.0,0.450,0.486,0.133,93.37,93.96,70.0,0.208,0.508,0.559,0.099,0.377,0.231,0.172,0.368,2,0,0,2,5,15,11,32,1,2,0.700,0.300,0.545,0.364,0.273,0.000,0.182,0.182,0.182,1.000,0.000,1.0,0.0,1.000,0.000,4,10,1,3,0.333,2,3,0.667,0,10,10,4,3,0,0,0,11,17.0,0.148,0.169,0.167,0.188,0.105,0.091,0.000,0.417,0.294,0.235,0.000,0.375,0.000,0.167,0.000,0.077,0.139
1,0021100005,1610612746,Los Angeles,Clippers,LAC,clippers,201933,Blake,Griffin,B. Griffin,blake-griffin,F,NaN,NaN,34:44,112.3,86.2,24.0,26.2,0.056,0.33,3.8,0.135,0.049,0.090,11.5,0.500,0.511,0.321,89.11,89.83,65.0,0.096,0.484,0.344,0.092,0.355,0.210,0.188,0.421,5,5,4,16,4,17,9,30,1,9,1.000,0.000,0.818,0.091,0.000,0.182,0.182,0.227,0.727,0.778,0.222,0.0,0.0,0.778,0.222,9,18,0,0,0.000,4,8,0.500,5,2,7,1,2,0,3,4,22,17.0,0.333,0.295,0.000,0.000,0.286,0.381,0.455,0.091,0.212,0.063,0.500,0.250,0.000,0.125,0.333,0.409,0.301
2,0021100005,1610612746,Los Angeles,Clippers,LAC,clippers,201599,DeAndre,Jordan,D. Jordan,deandre-jordan,C,NaN,NaN,30:31,111.9,86.4,20.3,25.4,0.000,0.00,0.0,0.088,0.054,0.070,20.0,0.500,0.412,0.139,93.18,92.80,59.0,0.006,0.500,0.569,0.115,0.357,0.232,0.173,0.424,0,1,2,2,5,15,9,26,1,6,1.000,0.000,0.333,0.000,0.000,0.333,0.667,0.000,0.333,1.000,0.000,0.0,0.0,1.000,0.000,1

In [5]:
df_games

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
0,22011,1610612746,LAC,Los Angeles Clippers,0021100005,2011-12-25,LAC @ GSW,W,240,105,37,81,0.457,8,23,0.348,23,38,0.605,13,30,43,22,9,8,10,19,19.0,2011-12
1,22011,1610612753,ORL,Orlando Magic,0021100004,2011-12-25,ORL @ OKC,L,240,89,30,81,0.370,8,28,0.286,21,26,0.808,14,31,45,18,7,4,18,25,-8.0,2011-12
2,22011,1610612747,LAL,Los Angeles Lakers,0021100003,2011-12-25,LAL vs. CHI,L,240,87,36,76,0.474,4,16,0.250,11,20,0.550,12,30,42,22,6,8,17,20,-1.0,2011-12
3,22011,1610612738,BOS,Boston Celtics,0021100001,2011-12-25,BOS @ NYK,L,240,104,39,76,0.513,2,5,0.400,24,31,0.774,13,28,41,28,7,5,18,28,-2.0,2011-12
4,22011,1610612744,GSW,Golden State Warriors,0021100005,2011-12-25,GSW vs. LAC,L,238,86,32,82,0.390,5,21,0.238,17,24,0.708,17,31,48,17,4,8,16,32,-19.0,2011-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35705,42024,1610612760,OKC,Oklahoma City Thunder,0042400315,2025-05-28,OKC vs. MIN,W,238,124,46,88,0.523,14,35,0.400,18,21,0.857,7,39,46,26,14,8,14,20,30.0,2024-25
35706,42024,1610612754,IND,Indiana Pacers,0042400305,2025-05-29,IND @ NYK,L,240,94,30,74,0.405,10,30,0.333,24,29,0.828,8,32,40,20,10,7,20,22,-17.0,2024-25
35707,42024,1610612752,NYK,New York Knicks,0042400305,2025-05-29,NYK vs. IND,W,241,111,44,89,0.494,8,29,0.276,15,22,0.682,11,34,45,22,11,3,15,22,17.0,2024-25
35708,42024,1610612752,NYK,New York Knicks,0042400306,2025-05-31,NYK @ IND,L,240,108,41,86,0.477,9,32,0.281,17,26,0.654,13,28,41,23,10,6,17,16,-17.0,2024-25


# 🧼 Nettoyage des minutes jouées

In [6]:
def convert_minutes(val):
    if pd.isna(val) or val in ['DNP', '']:
        return 0.0
    try:
        parts = str(val).split(':')
        return int(parts[0]) + int(parts[1]) / 60 if len(parts) == 2 else float(val)
    except:
        return 0.0

df_boxscores['MINUTES_PLAYED'] = df_boxscores['minutes'].apply(convert_minutes)

# Rename gameId and teamId in boxscores

In [7]:
rename_columns = {
    'gameId': 'GAME_ID',
    'teamId': 'TEAM_ID',
}

df_boxscores.rename(columns=rename_columns, inplace=True)


# 🔄 Cast dynamique des colonnes numériques


In [8]:
numeric_cols = df_boxscores.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    df_boxscores[col] = pd.to_numeric(df_boxscores[col], errors='coerce').fillna(0)

In [9]:
df_boxscores

,GAME_ID,TEAM_ID,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,MINUTES_PLAYED
0,0021100005,1610612746,Los Angeles,Clippers,LAC,clippers,2406,Caron,Butler,C. Butler,caron-butler,F,NaN,0.0,36:01,112.9,87.3,22.9,25.5,0.174,0.00,26.7,0.000,0.238,0.125,0.0,0.450,0.486,0.133,93.37,93.96,70.0,0.208,0.508,0.559,0.099,0.377,0.231,0.172,0.368,2,0,0,2,5,15,11,32,1,2,0.700,0.300,0.545,0.364,0.273,0.000,0.182,0.182,0.182,1.000,0.000,1.0,0.0,1.000,0.000,4,10,1,3,0.333,2,3,0.667,0,10,10,4,3,0,0,0,11,17.0,0.148,0.169,0.167,0.188,0.105,0.091,0.000,0.417,0.294,0.235,0.000,0.375,0.000,0.167,0.000,0.077,0.139,36.016667
1,0021100005,1610612746,Los Angeles,Clippers,LAC,clippers,201933,Blake,Griffin,B. Griffin,blake-griffin,F,NaN,0.0,34:44,112.3,86.2,24.0,26.2,0.056,0.33,3.8,0.135,0.049,0.090,11.5,0.500,0.511,0.321,89.11,89.83,65.0,0.096,0.484,0.344,0.092,0.355,0.210,0.188,0.421,5,5,4,16,4,17,9,30,1,9,1.000,0.000,0.818,0.091,0.000,0.182,0.182,0.227,0.727,0.778,0.222,0.0,0.0,0.778,0.222,9,18,0,0,0.000,4,8,0.500,5,2,7,1,2,0,3,4,22,17.0,0.333,0.295,0.000,0.000,0.286,0.381,0.455,0.091,0.212,0.063,0.500,0.250,0.000,0.125,0.333,0.409,0.301,34.733333
2,0021100005,1610612746,Los Angeles,Clippers,LAC,clippers,201599,DeAndre,Jordan,D. Jordan,deandre-jordan,C,NaN,0.0,30:31,111.9,86.4,20.3,25.4,0.000,0.00,0.0,0.088,0.054,0.070,20.0,0.500,0.412,0.139,93.18,92.80,59.0,0.006,0.500,0.569,0.115,0.357,0.232,0.173,0.424,0,1,2,2,5,15,9,26,1,6,1.000,0.000,0.333,0.000,0.000,0.333,0.667,0.000,0.

# ⚙️ Aggrégation par équipe et match


## Définir les colonnes à sommer et à moyenner pondérées


In [10]:

cols_to_sum = [
    'fieldGoalsMade_traditional', 'fieldGoalsAttempted_traditional',
    'threePointersMade_traditional', 'threePointersAttempted_traditional',
    'freeThrowsMade_traditional', 'freeThrowsAttempted_traditional',
    'reboundsOffensive_traditional', 'reboundsDefensive_traditional',
    'reboundsTotal_traditional', 'assists_traditional', 'steals_traditional',
    'blocks_traditional', 'turnovers_traditional', 'foulsPersonal_traditional',
    'points_traditional', 'MINUTES_PLAYED',
    'pointsOffTurnovers_misc', 'pointsSecondChance_misc', 'pointsFastBreak_misc',
    'pointsPaint_misc', 'blocksAgainst_misc'
]

cols_to_weighted_avg = [
    'offensiveRating_advanced', 'defensiveRating_advanced', 'netRating_advanced',
    'assistPercentage_advanced', 'assistToTurnover_advanced', 'assistRatio_advanced',
    'offensiveReboundPercentage_advanced', 'defensiveReboundPercentage_advanced',
    'reboundPercentage_advanced', 'turnoverRatio_advanced', 'effectiveFieldGoalPercentage_advanced',
    'trueShootingPercentage_advanced', 'usagePercentage_advanced', 'estimatedPace_advanced',
    'pace_advanced', 'possessions_advanced', 'PIE_advanced',
    'effectiveFieldGoalPercentage_fourfactors', 'freeThrowAttemptRate_fourfactors',
    'teamTurnoverPercentage_fourfactors', 'oppEffectiveFieldGoalPercentage_fourfactors',
    'oppFreeThrowAttemptRate_fourfactors', 'oppTeamTurnoverPercentage_fourfactors',
    'oppOffensiveReboundPercentage_fourfactors', 'foulsDrawn_misc',
    'percentageFieldGoalsAttempted2pt_scoring', 'percentageFieldGoalsAttempted3pt_scoring',
    'percentagePoints2pt_scoring', 'percentagePointsMidrange2pt_scoring',
    'percentagePoints3pt_scoring', 'percentagePointsFastBreak_scoring',
    'percentagePointsFreeThrow_scoring', 'percentagePointsOffTurnovers_scoring',
    'percentagePointsPaint_scoring', 'percentageAssisted2pt_scoring',
    'percentageUnassisted2pt_scoring', 'percentageAssisted3pt_scoring',
    'percentageUnassisted3pt_scoring', 'percentageAssistedFGM_scoring',
    'percentageUnassistedFGM_scoring', 'threePointersPercentage_traditional',
    'freeThrowsPercentage_traditional', 'percentageFieldGoalsMade_usage',
    'percentageFieldGoalsAttempted_usage', 'percentageThreePointersMade_usage',
    'percentageThreePointersAttempted_usage', 'percentageFreeThrowsMade_usage',
    'percentageFreeThrowsAttempted_usage', 'percentageReboundsOffensive_usage',
    'percentageReboundsDefensive_usage', 'percentageReboundsTotal_usage',
    'percentageAssists_usage', 'percentageTurnovers_usage', 'percentageSteals_usage',
    'percentageBlocks_usage', 'percentageBlocksAllowed_usage', 'percentagePersonalFouls_usage',
    'percentagePersonalFoulsDrawn_usage', 'percentagePoints_usage','plusMinusPoints_traditional'
]





## 📊 Agrégation des données par équipe et match


In [11]:

# 1. Agrégation par somme
group_keys = ['GAME_ID', 'TEAM_ID']
sum_agg = df_boxscores[group_keys + cols_to_sum].copy()
sum_agg = sum_agg.groupby(group_keys).sum().reset_index()

# 2. Agrégation pondérée par les minutes jouées
weighted_agg = compute_weighted_mean_features(df_boxscores, group_keys, cols_to_weighted_avg, weight_col='MINUTES_PLAYED')

# 3. Fusion des deux agrégats
team_match_stats = pd.merge(sum_agg, weighted_agg, on=group_keys, how='left')

# 4. Identifier l'équipe adverse
teams_in_game = df_boxscores.groupby('GAME_ID')['TEAM_ID'].unique().to_dict()
team_match_stats['OPP_TEAM_ID'] = team_match_stats.apply(
    lambda row: [tid for tid in teams_in_game[row['GAME_ID']] if tid != row['TEAM_ID']][0], axis=1
)


# Ajout GAME_DATE


In [12]:
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])
team_match_stats = team_match_stats.merge(
    df_games[['GAME_ID', 'TEAM_ID', 'GAME_DATE']],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [13]:

team_match_stats

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,GAME_DATE
0,0021100001,1610612738,78,152,4,10,48,62,26,56,82,56,14,10,36,56,208,480.066667,46,28,32,96,22,108.222233,111.932634,-3.694161,0.181532,0.790279,22.503020,0.061573,0.134002,0.097899,12.900889,0.453896,0.519213,0.191597,96.309683,95.611487,58.667477,0.105580,0.527782,0.418177,0.197031,0.535076,0.465478,0.165281,0.219223,3.478475,0.872849,0.063896,0.657995,0.198006,0.047118,0.093738,0.194901,0.231660,0.459988,0.644765,0.207825,0.157062,0.000000,0.662670,0.189920,0.104760,0.573335,0.193987,0.196816,0.157062,0.168254,0.215569,0.209100,0.174194,0.209209,0.198993,0.198954,0.194672,0.192078,0.164272,0.199008,0.197277,0.190504,0.198690,-0.024024,1610612752,2011-12-25
1,0021100001,1610612752,70,148,18,40,54,68,16,46,62,34,18,22,32,50,212,480.000000,36,30,16,60,10,111.690535,109.301778,2.391958,0.126097,0.960625,16.776063,0.036982,0.110483,0.073473,14.148785,0.504104,0.548899,0.195513,96.300708,95.199224,58.210000,0.093249,0.532830,0.466765,0.163975,0.523165,0.416090,0.198781,0.364813,3.618611,0.703230,0.259062,0.478796,0.197675,0.193396,0.055705,0.228786,0.154347,0.281354,0.426875,0.474098,0.536753,0.034983,0.528409,0.372563,0.320129,0.490452,0.193849,0.199039,0.163932,0.188751,0.187076,0.185633,0.201746,0.190901,0.189602,0.193623,0.204452,0.185782,0.199473,0.217573,0.210196,0.199814,0.195756,0.530694,1610612738,2011-12-25
2,0021100002,1610612742,62,164,18,56,46,60,16,46,62,46,20,0,34,58,188,480.000000,58,16,20,56,12,93.242326,103.892722,-10.645875,0.188613,0.796389,16.193778,0.030597,0.109217,0.064217,14.164944,0.391964,0.461304,0.197981,103.019453,100.899090,46.477153,0.117762,0.434031,0.369116,0.161

# 🔁 Merge avec l’adversaire

In [14]:
team_cols = [col for col in team_match_stats.columns if col not in ['GAME_ID', 'TEAM_ID', 'OPP_TEAM_ID']]
opp_cols = [f"OPP_{col}" for col in team_cols]
df_opp = team_match_stats.rename(columns={col: f"OPP_{col}" for col in team_cols}).rename(
    columns={'TEAM_ID': 'OPP_TEAM_ID', 'OPP_TEAM_ID': 'TEAM_ID'})
match_dataset = pd.merge(
    team_match_stats,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [15]:
match_dataset.columns

Index(['GAME_ID', 'TEAM_ID', 'fieldGoalsMade_traditional',
       'fieldGoalsAttempted_traditional', 'threePointersMade_traditional',
       'threePointersAttempted_traditional', 'freeThrowsMade_traditional',
       'freeThrowsAttempted_traditional', 'reboundsOffensive_traditional',
       'reboundsDefensive_traditional',
       ...
       'OPP_percentageAssists_usage', 'OPP_percentageTurnovers_usage',
       'OPP_percentageSteals_usage', 'OPP_percentageBlocks_usage',
       'OPP_percentageBlocksAllowed_usage',
       'OPP_percentagePersonalFouls_usage',
       'OPP_percentagePersonalFoulsDrawn_usage', 'OPP_percentagePoints_usage',
       'OPP_plusMinusPoints_traditional', 'OPP_GAME_DATE'],
      dtype='object', length=167)

In [16]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,GAME_DATE,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentage

# 🏠 Ajout IS_HOME et IS_WIN


In [17]:
# Assurer le format datetime pour GAME_DATE
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])

# Merge avec les infos de match (MATCHUP, SEASON)
match_dataset = match_dataset.merge(
    df_games[['GAME_ID', 'TEAM_ID', 'MATCHUP', 'SEASON']],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)

print("match_dataset after merge")
display(match_dataset)

# Définir si l'équipe joue à domicile
match_dataset['IS_HOME'] = match_dataset['MATCHUP'].str.contains('vs').astype(int)

# Calcul du résultat (win) et écart de points
match_dataset['IS_WIN'] = (match_dataset['points_traditional'] > match_dataset['OPP_points_traditional']).astype(int)
match_dataset['POINT_DIFF'] = match_dataset['points_traditional'] - match_dataset['OPP_points_traditional']

# Conversion GAME_DATE et tri
match_dataset['GAME_DATE'] = pd.to_datetime(match_dataset['GAME_DATE'])
match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)


match_dataset after merge


,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,GAME_DATE,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentage

In [18]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,GAME_DATE,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentage

# SPECIAL FOR ODDS : Keep only season past 2010-11


In [19]:
# # get first line with season 2010-11 and cut all lines before
# first_season = match_dataset[match_dataset['SEASON'] == '2010-11'].index[0]
# match_dataset = match_dataset.iloc[first_season:].reset_index(drop=True)
# match_dataset

# Merge Odds with dataset

In [20]:
all_odds_df = merge_odds_csv_files(DATA_ODDS_HISTORY_DIR)

match_dataset = match_odds_with_dataset(all_odds_df, match_dataset)



------------------ Nombre de lignes supprimées pour cotes manquantes: 14 ------------------


In [21]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,GAME_DATE,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentage

# Reorder columns for visualisation

In [22]:
cols_first = ['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'OPP_TEAM_ID', 'IS_HOME', 'IS_WIN','POINT_DIFF', 'points_traditional','ODDS','OPP_ODDS']
other_cols = [col for col in match_dataset.columns if col not in cols_first]
match_dataset = match_dataset[cols_first + other_cols]
match_dataset

,GAME_ID,GAME_DATE,TEAM_ID,OPP_TEAM_ID,IS_HOME,IS_WIN,POINT_DIFF,points_traditional,ODDS,OPP_ODDS,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_pe

# 🚀 Features avancées


In [28]:

# Ajouter les features avancées aux stats de match
#match_dataset = add_advanced_boxscore_features(match_dataset)

# MOVED TO CONFIG
# features_to_roll = cols_to_sum + cols_to_weighted_avg
# features_to_roll += [f"OPP_{col}" for col in cols_to_sum + cols_to_weighted_avg]

# Calcul des features glissantes shiftées
match_dataset = compute_rolling_features(match_dataset, "TEAM_ID", ["TEAM_ID", "GAME_DATE"], features_to_roll, N_LIST, method="ewm")


match_dataset = compute_winrates(match_dataset, "TEAM_ID", "IS_WIN", "IS_HOME", N_LIST)
match_dataset = compute_win_ratio(match_dataset, "TEAM_ID", "IS_WIN", N_LIST)

match_dataset["IS_WIN_SHIFTED"] = match_dataset.groupby("TEAM_ID")["IS_WIN"].shift(1).fillna(0).astype(int)
match_dataset["WIN_STREAK"] = match_dataset.groupby("TEAM_ID").apply(
    lambda x: compute_win_streak(x, "TEAM_ID", "IS_WIN_SHIFTED")).reset_index(level=0, drop=True)
match_dataset = compute_side_win_streak(match_dataset, win_shifted_col="IS_WIN_SHIFTED")


# Calcul des jours de repos pour l'équipe et l'adversaire
match_dataset["DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "TEAM_ID")
match_dataset["OPP_DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "OPP_TEAM_ID")

# Avantage de repos
match_dataset["REST_ADVANTAGE"] = (
    match_dataset["DAYS_SINCE_LAST_GAME"] - match_dataset["OPP_DAYS_SINCE_LAST_GAME"]
)


match_dataset = compute_rolling_rest_advantage(
    match_dataset, "TEAM_ID", "IS_HOME", "REST_ADVANTAGE", N_LIST
)

match_dataset = compute_home_away_pts(
    match_dataset, "TEAM_ID", "IS_HOME", "points_traditional", "OPP_points_traditional", N_LIST
)

# match_dataset = rename_pts_against_columns(match_dataset)

match_dataset = compute_h2h(match_dataset, N_LIST)
# match_dataset = compute_h2h_pts_margin(match_dataset, N_LIST)
match_dataset = compute_h2h_season(match_dataset)
match_dataset = compute_h2h_streak(match_dataset)

match_dataset = compute_elo(match_dataset)
match_dataset = compute_elo_season(match_dataset)


KeyboardInterrupt: 

In [24]:
pd.options.display.max_columns = None
display(match_dataset)

GAME_ID  GAME_DATE     TEAM_ID OPP_TEAM_ID  IS_HOME  IS_WIN  \
0      0021100001 2011-12-25  1610612738  1610612752      0.0     0.0   
1      0021100001 2011-12-25  1610612752  1610612738      1.0     1.0   
2      0021100002 2011-12-25  1610612742  1610612748      1.0     0.0   
3      0021100002 2011-12-25  1610612748  1610612742      0.0     1.0   
4      0021100003 2011-12-25  1610612741  1610612747      0.0     1.0   
...           ...        ...         ...         ...      ...     ...   
15231  0041600403 2017-06-07  1610612744  1610612739      0.0     1.0   
15232  0041600404 2017-06-09  1610612739  1610612744      1.0     1.0   
15233  0041600404 2017-06-09  1610612744  1610612739      0.0     0.0   
15234  0041600405 2017-06-12  1610612739  1610612744      0.0     0.0   
15235  0041600405 2017-06-12  1610612744  1610612739      1.0     1.0   

       POINT_DIFF  points_traditional  ODDS  OPP_ODDS  \
0            -4.0               208.0  2.45      1.37   
1             4.0               212.0  1.37      2.45   
2           -22.0               188.0  2.43      1.37   
3            22.0               210.0  1.37      2.43   
4             2.0               176.0  1.38      2.43   
...           ...                 ...   ...       ...   
15231         5.0               118.0  1.53      2.16   
15232        21.0               137.0  2.43      1.42   
15233       -21.0               116.0  1.42      2.43   
15234        -9.0               120.0  3.43      1.22   
15235         9.0               129.0  1.22      3.43   

       fieldGoalsMade_traditional  fieldGoalsAttempted_traditional  \
0                            78.0                            152.0   
1                            70.0                            148.0   
2                            62.0                            164.0   
3                            76.0                            156.0   
4                            72.0                            178.0   
...                           ...                              ...   
15231                        40.0                             83.0   
15232                        46.0                             87.0   
15233                        39.0                             87.0   
15234                        47.0                             88.0   
15235                        46.0                             90.0   

       threePointersMade_traditional  threePointersAttempted_traditional  \
0                                4.0                                10.0   
1                               18.0                                40.0   
2                               18.0                                56.0   
3                                8.0                                14.0   
4                               14.0                                30.0   
...                              ...                                 ...   
15231                           16.0                                33.0   
15232                           24.0                                45.0   
15233                           11.0                                39.0   
15234                           11.0                                24.0   
15235                           14.0                                38.0   

       freeThrowsMade_traditional  freeThrowsAttempted_traditional  \
0                            48.0                             62.0   
1                            54.0                             68.0   
2                            46.0                             60.0   
3                            50.0                             72.0   
4                            18.0                             28.0   
...                           ...                              ...   
15231                        22.0                             24.0   
15232                        21.0                             31.0   
15233                        27.0                             36

# 🧽 Nettoyage et sauvegarde


In [25]:


final_date = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
raw_save_path = os.path.join(DATA_FINAL_DATASET_DIR, f'nba_features_final_{final_date}.csv')
clean_save_path = os.path.join(DATA_FINAL_CLEANED_DATASET_DIR, f'nba_features_cleaned_final_{final_date}.csv')

os.makedirs(DATA_FINAL_DATASET_DIR, exist_ok=True)
os.makedirs(DATA_FINAL_CLEANED_DATASET_DIR, exist_ok=True)

match_dataset.to_csv(raw_save_path, index=False)

final_cleaned = match_dataset.drop(columns=features_to_roll+COLS_MATCH_REAL, errors='ignore')
final_cleaned.to_csv(clean_save_path, index=False)

print(f"✅ Fichier brut : {raw_save_path}")
print(f"✅ Fichier clean : {clean_save_path}")

final_cleaned

✅ Fichier brut : data\final_dataset\nba_features_final_2025-06-05_15-47-52.csv
✅ Fichier clean : data\final_cleaned_dataset\nba_features_cleaned_final_2025-06-05_15-47-52.csv


GAME_ID  GAME_DATE     TEAM_ID OPP_TEAM_ID  IS_HOME  IS_WIN  \
0      0021100001 2011-12-25  1610612738  1610612752      0.0     0.0   
1      0021100001 2011-12-25  1610612752  1610612738      1.0     1.0   
2      0021100002 2011-12-25  1610612742  1610612748      1.0     0.0   
3      0021100002 2011-12-25  1610612748  1610612742      0.0     1.0   
4      0021100003 2011-12-25  1610612741  1610612747      0.0     1.0   
...           ...        ...         ...         ...      ...     ...   
15231  0041600403 2017-06-07  1610612744  1610612739      0.0     1.0   
15232  0041600404 2017-06-09  1610612739  1610612744      1.0     1.0   
15233  0041600404 2017-06-09  1610612744  1610612739      0.0     0.0   
15234  0041600405 2017-06-12  1610612739  1610612744      0.0     0.0   
15235  0041600405 2017-06-12  1610612744  1610612739      1.0     1.0   

       POINT_DIFF  ODDS  OPP_ODDS   SEASON  ROLL_fieldGoalsMade_traditional_3  \
0            -4.0  2.45      1.37  2011-12                                NaN   
1             4.0  1.37      2.45  2011-12                                NaN   
2           -22.0  2.43      1.37  2011-12                                NaN   
3            22.0  1.37      2.43  2011-12                                NaN   
4             2.0  1.38      2.43  2011-12                                NaN   
...           ...   ...       ...      ...                                ...   
15231         5.0  1.53      2.16  2016-17                          45.943972   
15232        21.0  2.43      1.42  2016-17                          40.555353   
15233       -21.0  1.42      2.43  2016-17                          42.971986   
15234        -9.0  3.43      1.22  2016-17                          43.277676   
15235         9.0  1.22      3.43  2016-17                          40.985993   

       ROLL_fieldGoalsMade_traditional_5  ROLL_fieldGoalsMade_traditional_10  \
0                                    NaN                                 NaN   
1                                    NaN                                 NaN   
2                                    NaN                                 NaN   
3                                    NaN                                 NaN   
4                                    NaN                                 NaN   
...                                  ...                                 ...   
15231                          45.633276                           44.726909   
15232                          40.518146                           40.437527   
15233                          43.755517                           43.867471   
15234                          42.345430                           41.448886   
15235                          42.170345                           42.982477   

       ROLL_fieldGoalsMade_traditional_25  ROLL_fieldGoalsMade_traditional_50  \
0                                     NaN                                 NaN   
1                                     NaN                                 NaN   
2                                     NaN                                 NaN   
3                                     NaN                                 NaN   
4                                     NaN                                 NaN   
...                                   ...                                 ...   
15231                           43.599213                           43.078981   
15232                           40.459923                           40.429868   
15233                           43.322350                           42.958236   
15234                           40.886083                           40.648305   
15235                           42.989862                           42.803012   

       ROLL_fieldGoalsMade_traditional_100  \
0                                      NaN   
1                                      NaN   
2                                      NaN   
3                                      NaN

In [26]:
final_cleaned.columns

Index(['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'OPP_TEAM_ID', 'IS_HOME', 'IS_WIN',
       'POINT_DIFF', 'ODDS', 'OPP_ODDS', 'SEASON',
       ...
       'H2H_LAST_200_WINRATE', 'H2H_LAST_200_COUNT', 'H2H_SEASON_WINS',
       'H2H_SEASON_MATCHES', 'H2H_SEASON_WINRATE', 'H2H_WIN_STREAK', 'ELO_PRE',
       'OPP_ELO_PRE', 'ELO_PRE_SEASON', 'OPP_ELO_PRE_SEASON'],
      dtype='object', length=1242)

In [27]:
#store end time of notebook
end_time = datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-06-05 15:48:48.432251
Total time:  0:01:54.779310
